# 01 - Province Quarter Summary (Gold)

**Grain:** one row per province, year and quarter.

- Production is summed (total, and split by crop group)
- Climate columns are averaged, because they repeat on every crop row
- Risk labels are counted

Used by the dashboard.

In [0]:
from pyspark.sql import functions as F

from common.delta_io import write_delta_table
from configs.etl_config import SILVER_CROP_WEATHER_TABLE, GOLD_PROVINCE_SUMMARY_TABLE

enriched_df = spark.table(SILVER_CROP_WEATHER_TABLE)

summary_df = (
    enriched_df.groupBy("province", "year", "quarter")
    .agg(
        F.round(F.sum("production"), 3).alias("total_production"),
        F.round(F.sum(F.when(F.col("crop_group") == "PALAY", F.col("production"))), 3).alias("palay_production"),
        F.round(F.sum(F.when(F.col("crop_group") == "CORN", F.col("production"))), 3).alias("corn_production"),
        F.round(F.sum(F.when(F.col("crop_group") == "OTHER", F.col("production"))), 3).alias("other_production"),
        F.round(F.avg("temperature_2m_mean"), 2).alias("avg_temperature"),
        F.round(F.avg("precipitation_sum"), 2).alias("avg_precipitation"),
        F.round(F.avg("climate_stress"), 3).alias("climate_stress"),
        F.count(F.when(F.col("risk_label") == "Stable", 1)).alias("stable_count"),
        F.count(F.when(F.col("risk_label") == "Risk-prone", 1)).alias("risk_prone_count"),
        F.count(F.when(F.col("risk_label") == "High", 1)).alias("high_count"),
        F.count(F.when(F.col("risk_label") == "Declining", 1)).alias("declining_count"),
        F.countDistinct("crop").alias("crop_count"),
    )
    .fillna(0, subset=["palay_production", "corn_production", "other_production"])
)

write_delta_table(summary_df, GOLD_PROVINCE_SUMMARY_TABLE)
print(f"Rows: {spark.table(GOLD_PROVINCE_SUMMARY_TABLE).count():,}")